<img src="https://wamugii.kento.rw/logo-emblem-128.png" width="56" align="right" alt="WAMUGII">

**Machine Learning Fundamentals** · Week 3 · Evaluating and Improving Models · Lesson 15 of 24

# ROC Curves and Choosing a Threshold

**How to use this notebook:** run the code cells **in order**, from top to bottom (press ▶ on each cell, or use **Runtime → Run all**). You can change any code and run it again. To keep your changes, use **File → Save a copy in Drive**.

The lesson, quiz and certificate are on [WAMUGII](https://wamugii.kento.rw/courses/machine-learning-fundamentals).

**Learning objective:** Move the decision threshold to trade precision for recall, read a ROC curve and AUC, and choose a threshold from real business costs.

## The threshold is a business decision

`predict()` says "default" when the probability is above **0.5**. Nothing forces that. A lender who fears losses can refuse anyone above 0.3; one who wants more customers can use 0.7.

In [ ]:
import numpy as np
import pandas as pd

rng = np.random.default_rng(11)
n = 2000
income = rng.lognormal(12.2, 0.45, n).round(-3)
loan = rng.uniform(20_000, 300_000, n).round(-3)
months = rng.integers(1, 61, n)
late = rng.poisson(0.6, n)
salary = rng.random(n) < 0.55
z = -3.2 + 2.5 * (loan / income) - 0.03 * months + 0.8 * late - 0.9 * salary
default = (rng.random(n) < 1 / (1 + np.exp(-z))).astype(int)
loans = pd.DataFrame({"income_rwf": income, "loan_rwf": loan, "months_customer": months,
                      "late_payments": late, "salaried": salary.astype(int),
                      "defaulted": default})
print(loans.shape, "default rate:", round(loans["defaulted"].mean(), 3))
loans.head()

In [ ]:
from sklearn.model_selection import train_test_split

X = loans.drop(columns="defaulted")
y = loans["defaulted"]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y)

In [ ]:
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

model = make_pipeline(StandardScaler(), LogisticRegression())
model.fit(X_train, y_train)

In [ ]:
from sklearn.metrics import confusion_matrix, precision_score, recall_score

proba = model.predict_proba(X_test)[:, 1]
for t in [0.5, 0.4, 0.3, 0.2, 0.1]:
    pred_t = (proba >= t).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_test, pred_t).ravel()
    print(f"threshold {t}: caught {tp} of 106 defaulters, wrongly refused {fp}, "
          f"precision {precision_score(y_test, pred_t):.2f}, recall {recall_score(y_test, pred_t):.2f}")

Lowering the threshold from 0.5 to 0.2 raises recall from about **0.49 to 0.84** (89 of 106 defaulters caught) — but wrongly refuses 110 good borrowers instead of 28. Every threshold is a different trade-off.

## The ROC curve

The **ROC curve** shows every threshold at once:

- x-axis: **false positive rate** = FP ÷ (FP + TN) — good borrowers wrongly refused.
- y-axis: **true positive rate** = recall — defaulters caught.

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import RocCurveDisplay, roc_auc_score

RocCurveDisplay.from_predictions(y_test, proba, name="Logistic regression")
plt.plot([0, 1], [0, 1], "--", color="grey", label="random guessing")
plt.legend()
plt.show()
print("AUC:", round(roc_auc_score(y_test, proba), 3))

- A perfect model goes straight up to the top-left corner.
- The diagonal is random guessing.
- The **AUC** (area under the curve) summarises the whole curve: 1.0 is perfect, 0.5 is random. Our model has an AUC of about **0.86**.

AUC has a friendly meaning: pick one random defaulter and one random good borrower — AUC is the chance the model gives the defaulter the **higher** risk score.

## Compare models with AUC

In [ ]:
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(300, min_samples_leaf=5, random_state=42, n_jobs=-1).fit(X_train, y_train)
print("Random forest AUC:", round(roc_auc_score(y_test, rf.predict_proba(X_test)[:, 1]), 3))

The forest's AUC (about 0.84) is a little lower. AUC does not depend on any threshold, so it is a fair way to compare rankings. For very rare positives (fraud at 1%), also look at the **precision–recall curve** (`PrecisionRecallDisplay`), which is more sensitive to the rare class.

## Choose the threshold with money

Suppose the lender **earns a 15% fee** on each repaid loan and **loses 60%** of each defaulted loan (after some recovery). Let's compute the profit for every threshold on the test borrowers:

In [ ]:
import numpy as np

amount = X_test["loan_rwf"].to_numpy()
actual = y_test.to_numpy()

def profit(threshold):
    approve = proba < threshold
    earned = 0.15 * amount[approve & (actual == 0)].sum()
    lost = 0.60 * amount[approve & (actual == 1)].sum()
    return earned - lost

for t in [0.15, 0.25, 0.35, 0.5, 0.7, 1.01]:
    print(f"threshold {t}: profit {profit(t) / 1e6:.2f} million RWF")

- Approving **everyone** (threshold above 1) **loses** about 5.3 million RWF.
- The default threshold of 0.5 earns about 1.4 million.
- A threshold around **0.25** earns about **3.9 million** — almost three times more.

The best threshold came from the **costs**, not from accuracy. In a real project you choose it on validation data, then confirm it once on the test set.

## Key terms

- **Decision threshold** — the probability above which we predict the positive class.
- **ROC curve** — true positive rate against false positive rate for every threshold.
- **AUC** — area under the ROC curve; 0.5 = random, 1.0 = perfect.
- **Precision–recall curve** — precision against recall; better for rare positives.
- **Cost-based threshold** — the threshold that gives the best result in money or harm.

**Takeaway:** A model gives scores; people choose the threshold. ROC/AUC compares how well models rank cases, and real costs decide where to draw the line.

## Try it yourself

1. Plot the profit for thresholds from 0.05 to 0.95 in steps of 0.05. Where is the peak?
2. Change the costs: a 25% fee and a 50% loss. How does the best threshold move? Why?
3. Draw `PrecisionRecallDisplay.from_predictions(y_test, proba)` and describe its shape.
4. Write a short note to the lender's manager recommending a threshold, in plain words and RWF.

---
Finished? Go back to [Machine Learning Fundamentals on WAMUGII](https://wamugii.kento.rw/courses/machine-learning-fundamentals), tick the lesson as done and take the quiz.

*© WAMUGII TECH SOLUTIONS · Original course material for WAMUGII AI Program students.*